# Demo 06b — k-Nearest Neighbours

**Class 06 · Blocks 5-8** (slides: `slides/06_trees_knn.md`)

Six live demos, each with more experiments than the slides show:

* **K1 — KNN from scratch** with `jax.numpy`: pairwise distances, the $k$ nearest, the vote.
* **K2 — Scaling and $k$:** the same data with and without standardization; the U-shaped curve of $k$; weights.
* **K3 — The curse of dimensionality:** distances concentrate; irrelevant features ruin KNN.
* **K4 — KNN for regression** and its failure outside the data.
* **K5 — Cost, and all the learners of Classes 03-06 on one protocol.**
* **K6 — Faster k-NN:** prune the data (Hart, Wilson) and approximate the search (FAISS: IVF, HNSW, product quantization).

In [ ]:
import time
import warnings
from pathlib import Path

import faiss
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats
from sklearn.datasets import make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from faa import plotlib

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
DATA = Path("../../datasets")


plotlib.setup(module="06")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))


def table(rows, cols):
    keep = {"k"}
    rows = [tuple(f"{v:g}" if c in keep and isinstance(v, float) else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: null -> median / "missing", text -> one-hot."""
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy(), X.columns


Xb, yb, names_b = load_table("breast_cancer", "target")
Xd, yd, _ = load_table("digits", "target")
yd = yd.astype(int)
Xg, cg, _ = load_table("german_credit", "class")
yg = (cg == "bad").astype(int)
Xw, qw, _ = load_table("wine_quality", "quality")
yw = (qw >= 6).astype(int)
Xh, nh, _ = load_table("heart_disease", "num")
yh = (nh == ">50_1").astype(int)
Xs, ys_, _ = load_table("spambase", "spam")
Xs = np.log1p(Xs)
ys_ = ys_.astype(int)
SETS = {
    "breast_cancer": (Xb, yb),
    "digits": (Xd / 16.0, yd),
    "spambase": (Xs, ys_),
    "wine (quality >= 6)": (Xw, yw),
    "german_credit": (Xg, yg),
    "heart_disease": (Xh, yh),
}

## K1 — KNN from scratch

**The idea.** To classify a new point, find the $k$ training points closest to it and let them vote. There is **no
training**: the model *is* the training set ("lazy learning"). All the work is in the *distance*.

$$d(a, b) = \lVert a - b\rVert_2 = \sqrt{\sum_j (a_j - b_j)^2}$$

With the trick of Class 05, $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a^\top b$, all pairwise squared
distances between $m$ test points and $n$ training points are one matrix product.

In [ ]:
def pairwise_sq_dists(A, B):
    return jnp.sum(A**2, 1)[:, None] + jnp.sum(B**2, 1)[None, :] - 2 * A @ B.T


def knn_predict(Xtr, ytr, Xte, k=5, n_classes=2, weights="uniform"):
    d2 = jnp.maximum(pairwise_sq_dists(jnp.asarray(Xte), jnp.asarray(Xtr)), 0.0)
    neg, idx = jax.lax.top_k(-d2, k)  # the k smallest distances
    votes = jax.nn.one_hot(jnp.asarray(ytr)[idx], n_classes)  # (m, k, K)
    if weights == "distance":
        w = 1.0 / (jnp.sqrt(-neg) + 1e-9)
        votes = votes * w[:, :, None]
    return np.asarray(votes.sum(1).argmax(1))


rows = []
for name, (X, y) in SETS.items():
    K = len(np.unique(y))
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    sc = StandardScaler().fit(a)
    a, b = sc.transform(a), sc.transform(b)
    t0 = time.time()
    ours = knn_predict(a, ya, b, k=5, n_classes=K)
    t_ours = time.time() - t0
    ref = KNeighborsClassifier(5).fit(a, ya)
    rows.append(
        (name, float(np.mean(ours == yb_)), float(ref.score(b, yb_)), float(np.mean(ours == ref.predict(b))), t_ours)
    )
table(rows, ["dataset (k=5, standardized)", "acc ours", "acc sklearn", "same pred.", "our time (s)"])

A tiny worked example (the one on the slides): 6 cases A to F in 2D, a query at $(2.6, 1.2)$, $k = 1, 3, 5$; with
$k = 5$ the plain vote and the vote weighted by $1/d$ disagree.

In [ ]:
# slide: Worked Example: Six Cases, One Query
slide("Worked Example: Six Cases, One Query")
P = np.array([[1.0, 1.0], [1.0, 3.0], [3.0, 1.0], [3.0, 3.0], [5.0, 5.0], [6.0, 4.0]])
names6 = list("ABCDEF")
labels = np.array([0, 0, 1, 1, 1, 0])
q = np.array([2.6, 1.2])
d = np.sqrt(((P - q) ** 2).sum(1))
order = np.argsort(d)
table(
    [(rank + 1, names6[i], f"({P[i, 0]:g}, {P[i, 1]:g})", int(labels[i]), float(d[i])) for rank, i in enumerate(order)],
    ["rank", "case", "position", "class", "distance to (2.6, 1.2)"],
)
for k in (1, 3, 5):
    nn = order[:k]
    votes_ = np.bincount(labels[nn], minlength=2)
    print(f"k = {k}: neighbours {[names6[i] for i in nn]}, votes {votes_.tolist()} -> class {votes_.argmax()}")

In [ ]:
# plot k1_six
fg, ax = plotlib.fig("col", 1.5)
for i, (x_, y_) in enumerate(P):
    ax.scatter(x_, y_, s=22, color=ORANGE if labels[i] else BLUE, marker="s" if labels[i] else "o", zorder=3)
    ax.annotate(f"{names6[i]} {d[i]:.3f}", (x_, y_), (4, 3), textcoords="offset points", fontsize=7)
for i in order[:3]:
    ax.plot([q[0], P[i, 0]], [q[1], P[i, 1]], color=GREY, lw=0.8, zorder=1)
ax.scatter(*q, marker="*", s=60, color=RED, zorder=4)
ax.annotate("query", q, (-10, -13), textcoords="offset points", fontsize=7.5, color=RED)
ax.set(xlim=(0, 7), ylim=(0.2, 5.6), xlabel="$x_1$", ylabel="$x_2$")
ax.set_aspect("equal")
plotlib.save(fg, "k1_six")

In [ ]:
# slide: Weighted Voting
slide("Weighted Voting")
w6 = 1.0 / d[order[:5]]
cls5 = labels[order[:5]]
print("k = 5, weights 1/d:", {names6[i]: round(float(w), 2) for i, w in zip(order[:5], w6)})
print(
    f"class 1: {w6[cls5 == 1].sum():.2f}; class 0: {w6[cls5 == 0].sum():.2f} -> class {int(w6[cls5 == 1].sum() > w6[cls5 == 0].sum())} (the plain vote gave 0)"
)

**Other distances.** Euclidean ($p=2$), Manhattan ($p=1$), Chebyshev ($p=\infty$), cosine (angle), and Hamming (the number
of differing coordinates, for categorical data). The choice changes which points are neighbours:

In [ ]:
# slide: Distances
slide("Distances")
a_, b_ = np.array([1.0, 2.0]), np.array([4.0, 6.0])
dist_ab = {
    "Euclidean": float(np.linalg.norm(a_ - b_)),
    "Manhattan": float(np.abs(a_ - b_).sum()),
    "Chebyshev": float(np.abs(a_ - b_).max()),
    "cosine": float(1 - a_ @ b_ / (np.linalg.norm(a_) * np.linalg.norm(b_))),
    "Hamming": float((a_ != b_).sum()),
}
print({k: round(v, 3) for k, v in dist_ab.items()})
print(f"cosine: 1 - {a_ @ b_:.0f} / ({np.linalg.norm(a_):.2f} x {np.linalg.norm(b_):.2f}) = {dist_ab['cosine']:.3f}")

In [ ]:
# plot k1_distances
fg, axs = plotlib.fig("full", 1.45, ncols=3)
tt = np.linspace(0, 2 * np.pi, 200)
for ax, name in zip(axs, ("Euclidean", "Manhattan", "Chebyshev")):
    r_ = dist_ab[name]
    if name == "Euclidean":
        sx, sy = a_[0] + r_ * np.cos(tt), a_[1] + r_ * np.sin(tt)
    elif name == "Manhattan":
        sx, sy = a_[0] + np.array([r_, 0, -r_, 0, r_]), a_[1] + np.array([0, r_, 0, -r_, 0])
    else:
        sx, sy = a_[0] + r_ * np.array([1, 1, -1, -1, 1]), a_[1] + r_ * np.array([1, -1, -1, 1, 1])
    ax.plot(sx, sy, color=GREEN, lw=1.1)
    ax.scatter(*a_, color=BLUE, s=16, zorder=3)
    ax.scatter(*b_, color=RED, s=16, zorder=3)
    ax.annotate("$a$", a_, (-8, -9), textcoords="offset points", fontsize=7.5)
    ax.annotate("$b$", b_, (4, 3), textcoords="offset points", fontsize=7.5)
    ax.set_title(f"{name}: {r_:g}", fontsize=7.5, pad=3)
    ax.set(xlim=(-5, 8), ylim=(-4, 8), xticks=[], yticks=[])
    ax.set_aspect("equal")
    ax.grid(False)
plotlib.save(fg, "k1_distances")

In [ ]:
rows = []
for name in ("breast_cancer", "digits", "spambase", "heart_disease"):
    X, y = SETS[name]
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    sc = StandardScaler().fit(a)
    a, b = sc.transform(a), sc.transform(b)
    res = [name]
    for metric, kw in (("euclidean", {}), ("manhattan", {}), ("chebyshev", {}), ("cosine", {})):
        res.append(KNeighborsClassifier(5, metric=metric, algorithm="brute", **kw).fit(a, ya).score(b, yb_))
    rows.append(tuple(res))
table(rows, ["test accuracy, k=5", "euclidean", "manhattan", "chebyshev", "cosine"])

## K2 — Scaling and the choice of $k$

**Scaling.** The distance adds the squared differences of *all* features: a feature measured in large units decides the
neighbours. The standardization of Class 03 (training statistics only) is not optional.

In [ ]:
rows = []
for name, (X, y) in SETS.items():
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    raw = KNeighborsClassifier(5).fit(a, ya).score(b, yb_)
    sc = StandardScaler().fit(a)
    std = KNeighborsClassifier(5).fit(sc.transform(a), ya).score(sc.transform(b), yb_)
    spread = np.ptp(X, axis=0)
    rows.append((name, raw, std, float(spread.max() / max(spread[spread > 0].min(), 1e-12))))
table(rows, ["dataset", "acc, raw features", "acc, standardized", "range ratio max/min"])

In [ ]:
# slide: Why Scaling Is Essential
slide("Why Scaling Is Essential")
pa, pb = np.array([30.0, 30000.0]), np.array([40.0, 30500.0])
raw_terms = (pa - pb) ** 2
std_terms = ((pa - pb) / np.array([10.0, 10000.0])) ** 2
print(
    f"raw squared terms: age {raw_terms[0]:.0f}, income {raw_terms[1]:.0f}; age share {100 * raw_terms[0] / raw_terms.sum():.2f}%"
)
print(f"standardized (sd 10 years, 10000 euros): {std_terms[0]:.4f} + {std_terms[1]:.4f} = {std_terms.sum():.4f}")

**The choice of $k$** is the bias-variance trade-off of Class 01. $k = 1$: the boundary follows every point (low bias,
high variance, zero training error); a large $k$: a smooth boundary that approaches the majority class.

In [ ]:
Xm, ym = make_moons(300, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(3000, noise=0.3, random_state=1)
gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
rows = []
for ax, k in zip(axes, (1, 5, 15, 51, 201)):
    m = KNeighborsClassifier(k).fit(Xm, ym)
    z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, z, alpha=0.15, cmap="coolwarm")
    ax.scatter(*Xm.T, c=ym, cmap="coolwarm", s=8)
    ax.set_title(f"k = {k}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    rows.append((k, m.score(Xm, ym), m.score(Xm_te, ym_te)))
plt.tight_layout()
plt.show()
table(rows, ["k", "train accuracy", "test accuracy"])

In [ ]:
# plot k2_regions
slide("Choosing $k$: the Bias-Variance Knob")
fg, axs = plotlib.fig("full", 1.4, ncols=3)
for ax, k in zip(axs, (1, 15, 201)):
    m = KNeighborsClassifier(k).fit(Xm, ym)
    z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, z, levels=[-0.5, 0.5, 1.5], colors=[BLUE, ORANGE], alpha=0.22)
    ax.scatter(*Xm[ym == 0].T, s=5, color=BLUE, edgecolor="white", linewidth=0.2)
    ax.scatter(*Xm[ym == 1].T, s=5, color=ORANGE, edgecolor="white", linewidth=0.2)
    ax.set_title(f"$k$ = {k}\ntrain {m.score(Xm, ym):.3f}, test {m.score(Xm_te, ym_te):.3f}", fontsize=7.5, pad=3)
    ax.set(xticks=[], yticks=[])
    ax.grid(False)
plotlib.save(fg, "k2_regions")

In [ ]:
ks = np.arange(1, 61, 2)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
rows = []
for ax, name in zip(axes, ("breast_cancer", "digits", "german_credit")):
    X, y = SETS[name]
    accs = []
    for k in ks:
        pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(int(k)))
        accs.append(cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean())
    ax.plot(ks, accs)
    ax.set(xlabel="k", title=name)
    best = int(np.argmax(accs))
    rows.append((name, int(ks[best]), accs[best], accs[0], accs[-1]))
axes[0].set_ylabel("5-fold CV accuracy")
plt.tight_layout()
plt.show()
table(rows, ["dataset", "best k", "CV acc at best k", "CV acc, k=1", "CV acc, k=59"])

In [ ]:
# plot k2_curve
slide("The Curve of $k$")
k_list = [1, 3, 5, 9, 15, 25, 51, 101, 201]
curve = []
for k in k_list:
    m = KNeighborsClassifier(k).fit(Xm, ym)
    curve.append({"k": k, "train": float(m.score(Xm, ym)), "test": float(m.score(Xm_te, ym_te))})
curve = pl.DataFrame(curve)
show(curve)
fg, ax = plotlib.fig("full", 1.4)
ax.plot(curve["k"], curve["train"], color=BLUE, marker="o", ms=3, label="train")
ax.plot(curve["k"], curve["test"], color=RED, marker="s", ms=3, label="test")
ax.set(xscale="log", xlabel="$k$ (log scale)", ylabel="accuracy")
ax.set_xticks([1, 5, 15, 51, 201], ["1", "5", "15", "51", "201"])
ax.minorticks_off()
ax.legend(loc="lower left", fontsize=7.5)
plotlib.save(fg, "k2_curve")

**Weighting.** Instead of one vote per neighbour, weight each by $1/d$: close neighbours count more, ties become rare
and the dependence on $k$ weakens.

In [ ]:
rows = []
for name in ("breast_cancer", "digits", "spambase", "german_credit", "heart_disease"):
    X, y = SETS[name]
    res = [name]
    for k in (3, 15, 45):
        for w in ("uniform", "distance"):
            pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(k, weights=w))
            res.append(cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean())
    rows.append(tuple(res))
table(rows, ["CV accuracy", "3 unif.", "3 dist.", "15 unif.", "15 dist.", "45 unif.", "45 dist."])

In [ ]:
# plot k2_weighted
slide("Weighted Voting")
spam = [r for r in rows if r[0] == "spambase"][0]
fg, ax = plotlib.fig("full", 1.3)
ks3 = [3, 15, 45]
ax.plot(ks3, spam[1::2], color=BLUE, marker="o", ms=3, label="uniform")
ax.plot(ks3, spam[2::2], color=ORANGE, marker="s", ms=3, label="distance-weighted")
for k, u, w in zip(ks3, spam[1::2], spam[2::2]):
    ax.text(k, u - 0.004, f"{u:.3f}", ha="center", va="top", fontsize=7, color=BLUE)
    ax.text(k, w + 0.002, f"{w:.3f}", ha="center", va="bottom", fontsize=7, color=ORANGE)
ax.set(xscale="log", xlabel="$k$ (log scale)", ylabel="5-fold CV accuracy (spam)", ylim=(0.905, 0.95))
ax.set_xticks(ks3, [str(k) for k in ks3])
ax.minorticks_off()
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, fontsize=7.5)
plotlib.save(fg, "k2_weighted")

## K3 — The curse of dimensionality

In high dimension all points are almost **equally far** from each other: the "nearest" neighbour is hardly nearer than the
farthest, and a neighbourhood that holds a fixed share of the data is no longer *local* (Class 01).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for d in (1, 2, 5, 10, 20, 50, 100, 500, 1000):
    Z = rng.uniform(size=(500, d))
    q = rng.uniform(size=(1, d))
    dist = np.sqrt(((Z - q) ** 2).sum(1))
    # edge length of a hypercube that holds 1% of uniform data
    rows.append((d, float(dist.min()), float(dist.max()), float(dist.min() / dist.max()), float(0.01 ** (1 / d))))
table(rows, ["dimension d", "nearest", "farthest", "nearest / farthest", "cube edge for 1% of the data"])

In [ ]:
# plot k3_curse
slide("The Curse of Dimensionality")
fg, axs = plotlib.fig("full", 1.4, ncols=2)
dims_ = [r[0] for r in rows]
axs[0].plot(dims_, [r[3] for r in rows], color=BLUE, marker="o", ms=3)
axs[0].set(xscale="log", xlabel="dimension $d$ (log scale)", ylabel="nearest / farthest distance", ylim=(0, 1.02))
axs[1].plot(dims_, [r[4] for r in rows], color=RED, marker="s", ms=3)
axs[1].set(xscale="log", xlabel="dimension $d$ (log scale)", ylabel="edge holding 1% of the data", ylim=(0, 1.02))
for ax in axs:
    ax.minorticks_off()
    ax.set_xticks([1, 10, 100, 1000], ["1", "10", "100", "1000"])
fg.subplots_adjust(wspace=0.35)
plotlib.save(fg, "k3_curse")

**Irrelevant features.** Add $r$ columns of pure noise to the breast-cancer data (standardized). Every learner is
trained and tested on the same split. KNN counts every column in the distance, so it degrades fastest; models that can
*weight* or *select* features resist.

In [ ]:
rng = np.random.default_rng(0)
a, b, ya, yb_ = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=0)
rows = []
for r in (0, 10, 50, 100, 300, 1000):
    A = np.c_[a, rng.normal(size=(len(a), r))]
    B = np.c_[b, rng.normal(size=(len(b), r))]
    sc = StandardScaler().fit(A)
    A, B = sc.transform(A), sc.transform(B)
    res = [r]
    for m in (
        KNeighborsClassifier(5),
        LogisticRegression(max_iter=2000),
        SVC(),
        DecisionTreeClassifier(min_samples_leaf=5, random_state=0),
        GaussianNB(),
    ):
        res.append(m.fit(A, ya).score(B, yb_))
    rows.append(tuple(res))
table(rows, ["noise features", "KNN k=5", "logistic", "SVM rbf", "tree", "naive Bayes"])

In [ ]:
# plot k3_noise
slide("Irrelevant Features")
fg, ax = plotlib.fig("full", 1.35)
learn = ["KNN, $k = 5$", "logistic", "SVM (RBF)", "tree", "naive Bayes"]
cols_ = [RED, BLUE, GREEN, ORANGE, GREY]
xpos_ = np.arange(len(rows))
for j, (lab, colour) in enumerate(zip(learn, cols_)):
    ax.plot(xpos_, [r[1 + j] for r in rows], color=colour, marker="o", ms=3, label=lab, lw=1.6 if j == 0 else 1.0)
ax.set_xticks(xpos_, [str(r[0]) for r in rows])
ax.set(xlabel="noise columns added to breast cancer", ylabel="test accuracy")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=5, fontsize=6.5, columnspacing=0.8)
plotlib.save(fg, "k3_noise")

## K4 — KNN for regression

The prediction is the **mean** (or the distance-weighted mean) of the $k$ neighbours' targets: a local average. It is smooth
for large $k$ and follows the noise for $k=1$, and like a tree it cannot extrapolate.

In [ ]:
rng = np.random.default_rng(0)
xs = np.sort(rng.uniform(0, 6, 80))
ys = np.sin(xs) + rng.normal(0, 0.25, 80)
xg = np.linspace(-1, 7, 400)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
rows = []
for ax, k in zip(axes, (1, 3, 10, 40)):
    m = KNeighborsRegressor(k).fit(xs[:, None], ys)
    ax.scatter(xs, ys, s=8)
    ax.plot(xg, m.predict(xg[:, None]), color="tab:red")
    ax.plot(xg, np.sin(xg), "k:", lw=0.8)
    ax.set_title(f"k = {k}", fontsize=9)
    rows.append(
        (
            k,
            float(np.sqrt(np.mean((m.predict(xs[:, None]) - ys) ** 2))),
            float(
                np.sqrt(np.mean((m.predict(xg[(xg > 0) & (xg < 6)][:, None]) - np.sin(xg[(xg > 0) & (xg < 6)])) ** 2))
            ),
        )
    )
plt.tight_layout()
plt.show()
table(rows, ["k", "train RMSE", "RMSE to the true curve (0 < x < 6)"])

In [ ]:
def load_california():
    X, y, names = load_table("california_housing", "MedHouseVal")
    for j, n in enumerate(names):
        if n in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
            X[:, j] = np.log(X[:, j])
    return X, y.astype(float)


rows = []
for name, target in (
    ("concrete", "strength"),
    ("abalone", "rings"),
    ("auto_mpg", "mpg"),
    ("california_housing", "MedHouseVal"),
):
    X, y = load_california() if name == "california_housing" else load_table(name, target)[:2]
    y = y.astype(float)
    if len(y) > 6000:
        idx = np.random.default_rng(0).choice(len(y), 6000, replace=False)
        X, y = X[idx], y[idx]
    bins = np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))
    res = {}
    for label, mk in (
        ("mean", None),
        ("linear", LinearRegression),
        ("knn k=1", lambda: KNeighborsRegressor(1)),
        ("knn k=5", lambda: KNeighborsRegressor(5)),
        ("knn k=15", lambda: KNeighborsRegressor(15)),
        ("knn k=15 weighted", lambda: KNeighborsRegressor(15, weights="distance")),
    ):
        errs = []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, bins):
            sc = StandardScaler().fit(X[tr])
            if mk is None:
                pred = np.full(len(te), y[tr].mean())
            else:
                pred = mk().fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))
            errs.append(np.sqrt(np.mean((pred - y[te]) ** 2)))
        res[label] = float(np.mean(errs))
    rows.append((name, *res.values()))
table(rows, ["RMSE (5-fold)", "mean", "linear", "knn k=1", "knn k=5", "knn k=15", "knn k=15 weighted"])

## K5 — Cost, and all the learners on one protocol

**Cost.** Training is free (store the data); **prediction** compares the query with every training point:
$O(n\,d)$ per query. Spatial trees (**kd-tree**, **ball tree**) prune most of the comparisons in low dimension, and stop
helping when $d$ grows (the curse again).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for n, d in ((2000, 2), (20000, 2), (20000, 10), (20000, 30), (20000, 100)):
    Xtr_ = rng.normal(size=(n, d))
    ytr_ = (Xtr_[:, 0] > 0).astype(int)
    Xq = rng.normal(size=(500, d))
    res = [n, d]
    for algo in ("brute", "kd_tree", "ball_tree"):
        m = KNeighborsClassifier(5, algorithm=algo)
        t0 = time.time()
        m.fit(Xtr_, ytr_)
        t_fit = time.time() - t0
        t0 = time.time()
        m.predict(Xq)
        res.append(time.time() - t0)
    rows.append(tuple(res))
table(rows, ["n", "d", "brute (s / 500 queries)", "kd-tree", "ball tree"])
print(
    f"memory of the model: n x d numbers; 1 000 000 examples x 100 features in float64 = {1_000_000 * 100 * 8 / 1e9:.1f} GB"
)

In [ ]:
# plot k5_cost
slide("The Cost of Prediction")
fg, ax = plotlib.fig("full", 1.3)
width = 0.26
for j, (lab, colour) in enumerate((("brute force", BLUE), ("kd-tree", GREEN), ("ball tree", ORANGE))):
    ax.bar(np.arange(len(rows)) + (j - 1) * width, [r[2 + j] for r in rows], width, color=colour, label=lab)
ax.set_xticks(np.arange(len(rows)), [f"{r[0]:,}\n$d$ = {r[1]}".replace(",", " ") for r in rows], fontsize=7)
ax.set(yscale="log", ylabel="seconds, 500 queries (log scale)", xlabel="training cases")
ax.legend(loc="upper left", fontsize=7.5, ncols=3)
plotlib.save(fg, "k5_cost")

**One protocol, six learners.** Logistic regression (Class 03), Gaussian naive Bayes (Class 04), RBF SVM (Class 05), a pruned
decision tree and KNN (this class), on 5 stratified 75/25 splits with the standardization fitted on the training part;
$k$ and the tree's `min_samples_leaf` chosen by an inner 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

learners = {
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "naive Bayes": lambda: make_pipeline(StandardScaler(), GaussianNB()),
    "SVM rbf": lambda: make_pipeline(StandardScaler(), SVC()),
    "tree": lambda: GridSearchCV(
        DecisionTreeClassifier(random_state=0), {"min_samples_leaf": [1, 5, 10, 20, 50]}, cv=5
    ),
    "KNN": lambda: GridSearchCV(
        make_pipeline(StandardScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [1, 3, 5, 9, 15, 25]},
        cv=5,
    ),
}
results = {}
timing = {}
for name, (X, y) in SETS.items():
    for rep in range(5):
        a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=rep)
        for lname, mk in learners.items():
            t0 = time.time()
            m = mk().fit(a, ya)
            timing.setdefault((name, lname), []).append(time.time() - t0)
            results.setdefault((name, lname), []).append(m.score(b, yb_))
rows = []
for name in SETS:
    row = [name]
    for lname in learners:
        row.append(float(np.median(results[(name, lname)])))
    rows.append(tuple(row))
table(rows, ["median test accuracy", *learners])
print()
table(
    [(lname, float(np.median([np.median(timing[(n, lname)]) for n in SETS]))) for lname in learners],
    ["learner", "median fit time incl. tuning (s)"],
)

**Is the difference real?** The corrected resampled $t$-test of Class 05 for KNN against the tree and the SVM:

In [ ]:
def corrected_ttest(a, b, ratio=1 / 3):
    d = np.asarray(a) - np.asarray(b)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / len(d) + ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), len(d) - 1))


rows = []
for name in SETS:
    for A, B in (("KNN", "tree"), ("KNN", "SVM rbf"), ("KNN", "logistic")):
        t, p = corrected_ttest(results[(name, A)], results[(name, B)])
        rows.append((name, f"{A} - {B}", float(np.mean(results[(name, A)]) - np.mean(results[(name, B)])), t, p))
table(rows, ["dataset", "comparison", "mean difference", "t", "p"])

In [ ]:
# plot k5_ttest
slide("Is a Difference Real?")
fg, ax = plotlib.fig("full", 1.45)
comps = ["KNN - tree", "KNN - SVM rbf", "KNN - logistic"]
names_ds = list(SETS)
for j, (comp, colour) in enumerate(zip(comps, (GREEN, ORANGE, BLUE))):
    sub = [r for r in rows if r[1] == comp]
    ypos_ = np.arange(len(sub))[::-1] + (1 - j) * 0.24
    ax.scatter(
        [r[2] for r in sub],
        ypos_,
        facecolors=[colour if r[4] < 0.05 else "white" for r in sub],
        edgecolors=colour,
        linewidths=1.0,
        s=18,
        zorder=3,
        label=comp,
    )
ax.axvline(0, color="#444444", lw=0.8)
ax.set_yticks(np.arange(len(names_ds))[::-1], [n.split(" (")[0].replace("_", " ") for n in names_ds])
ax.set(xlabel="difference of the mean accuracy (KNN minus the other); filled: $p < 0.05$", xlim=(-0.06, 0.15))
ax.legend(loc="lower right", fontsize=7)
plotlib.save(fg, "k5_ttest")

## K6 — Faster k-NN: prune the data, approximate the search

k-NN compares a query with **every** stored case. Two ways to cut that cost:

* **Prototype selection** (pruning): keep only the cases that matter. Hart's *condensed nearest neighbour* (1968) keeps
  the cases that the stored ones cannot classify; Wilson's *edited nearest neighbour* (1972) first deletes the cases whose
  class disagrees with their neighbours (noise).
* **Approximate nearest neighbours (ANN)**: search a structure (cells, a graph, compressed vectors) instead of all cases,
  and accept that the neighbours found are *almost* the nearest. FAISS (Meta) is the reference library.

In [ ]:
# slide: Prune the Data: Keep the Cases That Matter
slide("Prune the Data: Keep the Cases That Matter")


def condense(X, y, seed):
    """Hart (1968), condensed nearest neighbour: keep one case, then add every case the kept cases misclassify (1-NN); sweep until a pass adds nothing."""
    order = np.random.default_rng(seed).permutation(len(X))
    keep = [int(order[0])]
    in_store = np.zeros(len(X), bool)
    in_store[order[0]] = True
    passes, added = 0, True
    while added:
        added = False
        passes += 1
        for i in order:
            if in_store[i]:
                continue
            nearest = keep[int(np.argmin(np.linalg.norm(X[keep] - X[i], axis=1)))]
            if y[nearest] != y[i]:
                keep.append(int(i))
                in_store[i] = True
                added = True
    return np.array(keep), passes


def edit(X, y, k=3):
    """Wilson (1972), edited nearest neighbour: delete every case whose class differs from the majority of its k nearest other cases."""
    nn = (
        KNeighborsClassifier(k + 1).fit(X, y).kneighbors(X, return_distance=False)[:, 1:]
    )  # the first neighbour is the case itself
    votes = np.array([np.bincount(y[row], minlength=y.max() + 1).argmax() for row in nn])
    return np.flatnonzero(votes == y)


Xm2, ym2 = make_moons(300, noise=0.25, random_state=6)
Xt2, yt2 = make_moons(4000, noise=0.25, random_state=60)
keep2, passes2 = condense(Xm2, ym2, seed=6)
full1 = KNeighborsClassifier(1).fit(Xm2, ym2)
kept1 = KNeighborsClassifier(1).fit(Xm2[keep2], ym2[keep2])
print(f"two moons, n = 300: Hart keeps {len(keep2)} cases ({len(keep2) / 300:.3f}) after {passes2} passes")
print(
    f"1-NN test error on 4000 fresh points: all cases {1 - full1.score(Xt2, yt2):.3f}, kept cases {1 - kept1.score(Xt2, yt2):.3f}"
)
print(
    f"the two classifiers give the same answer on {np.mean(full1.predict(Xt2) == kept1.predict(Xt2)):.3f} of the fresh points"
)
edited2 = edit(Xm2, ym2)
print(
    f"Wilson (k = 3) removes {300 - len(edited2)} noisy cases; Hart on the edited set keeps {len(condense(Xm2[edited2], ym2[edited2], 6)[0])}"
)

In [ ]:
# plot k6_condense
gx2, gy2 = np.meshgrid(np.linspace(-1.6, 2.6, 420), np.linspace(-1.1, 1.6, 260))
G2 = np.c_[gx2.ravel(), gy2.ravel()]
border_all = full1.predict(G2).reshape(gx2.shape)
border_kept = kept1.predict(G2).reshape(gx2.shape)
fg, axs = plotlib.fig("full", 1.4, ncols=2)
for ax, title, model_grid in (
    (axs[0], f"all {len(Xm2)} cases", border_all),
    (axs[1], f"{len(keep2)} kept cases", border_kept),
):
    ax.contourf(gx2, gy2, model_grid, levels=[-0.5, 0.5, 1.5], colors=[BLUE, ORANGE], alpha=0.2)
    ax.contour(gx2, gy2, model_grid, levels=[0.5], colors=["#222222"], linewidths=0.8)
axs[0].scatter(*Xm2[ym2 == 0].T, s=6, color=BLUE, edgecolor="white", linewidth=0.2)
axs[0].scatter(*Xm2[ym2 == 1].T, s=6, color=ORANGE, edgecolor="white", linewidth=0.2)
axs[1].scatter(*Xm2.T, s=4, color=plotlib.LIGHT, zorder=2)
axs[1].scatter(*Xm2[keep2][ym2[keep2] == 0].T, s=12, color=BLUE, edgecolor="black", linewidth=0.5, zorder=3)
axs[1].scatter(*Xm2[keep2][ym2[keep2] == 1].T, s=12, color=ORANGE, edgecolor="black", linewidth=0.5, zorder=3)
axs[1].contour(gx2, gy2, border_all, levels=[0.5], colors=[GREY], linewidths=0.8, linestyles="--")
for ax, title in zip(
    axs, (f"all {len(Xm2)} cases", f"{len(keep2)} kept cases (large), {len(Xm2) - len(keep2)} removed (grey)")
):
    ax.set(xlim=(-1.6, 2.6), ylim=(-1.1, 1.6), xticks=[], yticks=[])
    ax.set_title(title, fontsize=7.5, pad=3)
    ax.grid(False)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set(visible=True, color="#999999", linewidth=0.6)
plotlib.save(fg, "k6_condense")

In [ ]:
# slide: Pruning in Numbers
slide("Pruning in Numbers")


def predict_seconds(model, X, repeats=5):
    """Median wall time of one batch of predictions (typical value of this machine)."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        model.predict(X)
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def prune_study(name, seed=0, fractions=(0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0), repeats=6):
    X, y = SETS[name]
    A, B, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    sc = StandardScaler().fit(A)
    A, B = sc.transform(A), sc.transform(B)
    brute = lambda: KNeighborsClassifier(1, algorithm="brute")  # noqa: E731
    rows = []
    full = brute().fit(A, ya)
    rows.append(
        {
            "data set": name,
            "method": "all cases",
            "fraction": 1.0,
            "accuracy": float(full.score(B, yb)),
            "sd": 0.0,
            "seconds": predict_seconds(full, B),
        }
    )
    rng = np.random.default_rng(seed)
    for f in fractions[:-1]:
        accs, secs = [], []
        for _ in range(repeats):
            idx = rng.choice(len(A), max(2, int(f * len(A))), replace=False)
            m = brute().fit(A[idx], ya[idx])
            accs.append(m.score(B, yb))
            secs.append(predict_seconds(m, B, 3))
        rows.append(
            {
                "data set": name,
                "method": "random subset",
                "fraction": f,
                "accuracy": float(np.mean(accs)),
                "sd": float(np.std(accs)),
                "seconds": float(np.median(secs)),
            }
        )
    keep, _ = condense(A, ya, seed)
    m = brute().fit(A[keep], ya[keep])
    rows.append(
        {
            "data set": name,
            "method": "Hart (CNN)",
            "fraction": len(keep) / len(A),
            "accuracy": float(m.score(B, yb)),
            "sd": 0.0,
            "seconds": predict_seconds(m, B),
        }
    )
    ed = edit(A, ya)
    m = brute().fit(A[ed], ya[ed])
    rows.append(
        {
            "data set": name,
            "method": "Wilson (ENN)",
            "fraction": len(ed) / len(A),
            "accuracy": float(m.score(B, yb)),
            "sd": 0.0,
            "seconds": predict_seconds(m, B),
        }
    )
    keep2_, _ = condense(A[ed], ya[ed], seed)
    both = ed[keep2_]
    m = brute().fit(A[both], ya[both])
    rows.append(
        {
            "data set": name,
            "method": "Wilson, then Hart",
            "fraction": len(both) / len(A),
            "accuracy": float(m.score(B, yb)),
            "sd": 0.0,
            "seconds": predict_seconds(m, B),
        }
    )
    return rows


t0 = time.time()
prune_rows = []
for name in ("digits", "spambase", "breast_cancer", "wine (quality >= 6)"):
    prune_rows += prune_study(name)
prune = pl.DataFrame(prune_rows)
print(
    prune.filter(pl.col("method") != "random subset")
    .with_columns(
        pl.col("fraction", "accuracy", "sd").round(3), (pl.col("seconds") * 1e3).round(2).alias("ms per test set")
    )
    .drop("seconds")
)
d_rnd = prune.filter((pl.col("data set") == "digits") & (pl.col("method") == "random subset"))
print("digits, random subsets:", ", ".join(f"{r['fraction']:.2f}: {r['accuracy']:.3f}" for r in d_rnd.to_dicts()))
print(f"cell time {time.time() - t0:.0f} s; times are typical values of one machine, brute-force search")

In [ ]:
# plot k6_prune
fg, axs = plotlib.fig("full", 1.45, ncols=2)
d6 = prune.filter(pl.col("data set") == "digits")
d6 = d6.with_columns((pl.col("seconds") * 1e3).alias("ms"))
rnd = d6.filter(pl.col("method") == "random subset")
for ax, key, label in ((axs[0], "accuracy", "1-NN test accuracy"), (axs[1], "ms", "ms (log scale)")):
    ax.plot(rnd["fraction"], rnd[key], color=GREY, marker="o", ms=3, label="random subset")
    if key == "accuracy":
        ax.fill_between(
            rnd["fraction"],
            rnd["accuracy"] - rnd["sd"],
            rnd["accuracy"] + rnd["sd"],
            color=GREY,
            alpha=0.2,
            linewidth=0,
        )
    full_row = d6.filter(pl.col("method") == "all cases")
    ax.plot(full_row["fraction"], full_row[key], color=GREY, marker="o", ms=3)
    for method, colour, marker in (
        ("Hart (CNN)", RED, "D"),
        ("Wilson (ENN)", PURPLE, "s"),
        ("Wilson, then Hart", BLUE, "^"),
    ):
        r = d6.filter(pl.col("method") == method)
        ax.plot(r["fraction"], r[key], color=colour, marker=marker, ms=5, ls="none", label=method)
    ax.set(xscale="log", xlabel="fraction kept (log scale)", ylabel=label)
    ax.set_xticks([0.02, 0.1, 0.5, 1.0], ["0.02", "0.1", "0.5", "1"])
    ax.minorticks_off()
axs[1].set_yscale("log")
axs[1].set_yticks([0.5, 1, 2, 5], ["0.5", "1", "2", "5"])
axs[1].minorticks_off()
handles, labels = axs[0].get_legend_handles_labels()
fg.legend(
    handles,
    labels,
    loc="upper center",
    ncols=4,
    bbox_to_anchor=(0.5, 1.07),
    fontsize=7,
    columnspacing=1.0,
    handletextpad=0.3,
)
fg.subplots_adjust(wspace=0.45)
plotlib.save(fg, "k6_prune")

In [ ]:
# slide: Approximate Nearest Neighbours
slide("Approximate Nearest Neighbours")
faiss.omp_set_num_threads(16)
N_BASE, N_QUERY, D_ANN, K_ANN, N_CLUSTERS = 200_000, 1_000, 64, 10, 400


def clustered(n, seed, centres=None):
    """n vectors of dimension 64 around 400 centres (real embeddings are clustered too); returns the data and the centres."""
    rng = np.random.default_rng(seed)
    if centres is None:
        centres = rng.normal(0, 0.7, size=(N_CLUSTERS, D_ANN)).astype("float32")
    label = rng.integers(0, N_CLUSTERS, n)
    return (centres[label] + rng.normal(0, 1.0, size=(n, D_ANN))).astype("float32"), centres


base, centres = clustered(N_BASE, seed=6)
queries, _ = clustered(N_QUERY, seed=7, centres=centres)
flat = faiss.IndexFlatL2(D_ANN)
flat.add(base)
faiss.omp_set_num_threads(1)  # one thread for every timing: the comparison is per core
t0 = time.perf_counter()
_, truth = flat.search(queries, K_ANN)
t_flat = time.perf_counter() - t0
print(f"{N_BASE} vectors x {D_ANN} dimensions, {N_QUERY} queries, k = {K_ANN}, one thread")
print(
    f"exact search (IndexFlatL2): {t_flat:.2f} s for {N_QUERY} queries = {N_QUERY / t_flat:.0f} queries/s; the stored vectors take {base.nbytes / 1e6:.0f} MB"
)


def recall_at_k(found, truth):
    """Share of the true k nearest neighbours that the approximate search returned (averaged over the queries)."""
    return float(np.mean([len(set(f) & set(t)) / len(t) for f, t in zip(found, truth)]))


def measure(index, label, setting, repeats=2):
    """Queries per second (best of `repeats`) and recall@k of an index."""
    best = np.inf
    for _ in range(repeats):
        t0 = time.perf_counter()
        _, found = index.search(queries, K_ANN)
        best = min(best, time.perf_counter() - t0)
    return {"index": label, "setting": setting, "recall@10": recall_at_k(found, truth), "queries/s": N_QUERY / best}


ann_rows = [{"index": "flat (exact)", "setting": "-", "recall@10": 1.0, "queries/s": N_QUERY / t_flat}]

In [ ]:
# slide: Approximate Nearest Neighbours
slide("Approximate Nearest Neighbours")
faiss.omp_set_num_threads(16)
t0 = time.perf_counter()
quantizer = faiss.IndexFlatL2(D_ANN)
ivf = faiss.IndexIVFFlat(quantizer, D_ANN, 512)
ivf.train(base)
ivf.add(base)
print(f"IVF with 512 cells: trained and filled in {time.perf_counter() - t0:.1f} s")
faiss.omp_set_num_threads(1)
for nprobe in (1, 2, 4, 8, 16, 32, 64):
    ivf.nprobe = nprobe
    ann_rows.append(measure(ivf, "IVF (512 cells)", f"nprobe = {nprobe}"))

In [ ]:
# slide: Approximate Nearest Neighbours
slide("Approximate Nearest Neighbours")
faiss.omp_set_num_threads(16)
t0 = time.perf_counter()
hnsw = faiss.IndexHNSWFlat(D_ANN, 32)
hnsw.hnsw.efConstruction = 100
hnsw.add(base)
print(f"HNSW (32 links per node): built in {time.perf_counter() - t0:.1f} s")
faiss.omp_set_num_threads(1)
for ef in (4, 8, 16, 32, 64, 128, 256):
    hnsw.hnsw.efSearch = ef
    ann_rows.append(measure(hnsw, "HNSW (M = 32)", f"efSearch = {ef}"))

In [ ]:
# slide: Approximate Nearest Neighbours
slide("Approximate Nearest Neighbours")
faiss.omp_set_num_threads(16)
t0 = time.perf_counter()
ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatL2(D_ANN), D_ANN, 512, 16, 8)
ivfpq.train(base)
ivfpq.add(base)
print(
    f"IVF + product quantization (16 bytes per vector): built in {time.perf_counter() - t0:.1f} s; the codes take {N_BASE * 16 / 1e6:.1f} MB instead of {base.nbytes / 1e6:.0f} MB"
)
faiss.omp_set_num_threads(1)
for nprobe in (1, 4, 16, 64):
    ivfpq.nprobe = nprobe
    ann_rows.append(measure(ivfpq, "IVF + PQ (16 bytes)", f"nprobe = {nprobe}"))
ann = pl.DataFrame(ann_rows)
flat_qps = float(ann["queries/s"][0])
print(
    ann.with_columns(
        pl.col("recall@10").round(3),
        pl.col("queries/s").round(0),
        (pl.col("queries/s") / flat_qps).round(1).alias("speed-up"),
    )
)
print("timings: typical values of one core of this machine")

In [ ]:
# plot k6_ann
fg, ax = plotlib.fig("full", 1.5)
for label, colour, marker in (
    ("HNSW (M = 32)", BLUE, "o"),
    ("IVF (512 cells)", GREEN, "s"),
    ("IVF + PQ (16 bytes)", ORANGE, "^"),
):
    sub = ann.filter(pl.col("index") == label)
    ax.plot(sub["recall@10"], sub["queries/s"], color=colour, marker=marker, ms=3, label=label)
ax.plot([1.0], [flat_qps], marker="D", color=RED, ms=5, ls="none", label="flat: exact")
ax.set(yscale="log", xlabel="recall@10 (share of the true 10 nearest found)", ylabel="queries per second (log scale)")
ax.set_xlim(0.35, 1.02)
ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncols=4, fontsize=6.5, columnspacing=0.8)
plotlib.save(fg, "k6_ann")

In [ ]:
# slide: How ANN Indexes Skip Distances
slide("How ANN Indexes Skip Distances")
faiss.omp_set_num_threads(16)
rng = np.random.default_rng(8)
uni = rng.uniform(size=(50_000, D_ANN)).astype("float32")
uq = rng.uniform(size=(500, D_ANN)).astype("float32")
flat_u = faiss.IndexFlatL2(D_ANN)
flat_u.add(uni)
_, truth_u = flat_u.search(uq, K_ANN)
ivf_u = faiss.IndexIVFFlat(faiss.IndexFlatL2(D_ANN), D_ANN, 256)
ivf_u.train(uni)
ivf_u.add(uni)
rows = []
for nprobe in (1, 8, 32, 128):
    ivf_u.nprobe = nprobe
    _, found = ivf_u.search(uq, K_ANN)
    rows.append(
        {
            "data": "uniform noise, 64-D",
            "nprobe": nprobe,
            "share of the cells searched": nprobe / 256,
            "recall@10": recall_at_k(found, truth_u),
        }
    )
ivf.nprobe = 8
_, found = ivf.search(queries, K_ANN)
rows.append(
    {
        "data": "clustered, 64-D",
        "nprobe": 8,
        "share of the cells searched": 8 / 512,
        "recall@10": recall_at_k(found, truth),
    }
)
show(pl.DataFrame(rows), 3)
print(
    "without structure the nearest neighbour hides in any cell (the curse of dimensionality again): searching a few cells finds almost none of the true neighbours"
)

## Summary

* KNN stores the data and answers with the **$k$ nearest** points: no training, a nonlinear boundary, an honest
  "like the ones I know".
* Everything hangs on the **distance**: standardize first; irrelevant features and high dimension destroy it.
* $k$ is the bias-variance knob: $k = 1$ memorizes; a large $k$ smooths; choose it on validation data.
* Prediction is $O(nd)$ per query; kd-trees help only in low dimension.
* Two ways to cut the cost: **prune** the stored cases (Hart keeps the ones near the border, 13-43% of the data here, a few points of accuracy lost) or **approximate** the search (FAISS: 10-100 times the speed of the exact search at a recall of 0.8-0.95).
* No learner wins everywhere: compare on the same protocol with a corrected test.